# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26452\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra columns for interaction features

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

### Creating column representing high age

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Accounting for multiple risks

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26452\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Create base random forest classifier model

In [42]:
base_classifier = RandomForestClassifier(random_state=42)

### Fitting base classifier on unsampled data

In [43]:
base_classifier.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [44]:
importances = base_classifier.feature_importances_

In [45]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)

In [46]:
print(feature_imp_df)

                           Feature  Gini Importance
24           age_avg_glucose_level         0.093389
4                avg_glucose_level         0.090977
33           log_avg_glucose_level         0.088880
23                         age_bmi         0.088791
25           bmi_avg_glucose_level         0.081161
0                              age         0.078885
32                         log_bmi         0.076191
5                              bmi         0.075715
20                age_ever_married         0.065753
21                age_hypertension         0.024019
26  avg_glucose_level_hypertension         0.023449
22               age_heart_disease         0.020540
31                      risk_count         0.019931
7                            urban         0.015392
13               work_type_private         0.013761
6                             male         0.013212
30                             old         0.012217
18     smoking_status_never_smoked         0.012056
8           

### Feature selection

In [47]:
# retained_columns = ['age_avg_glucose_level', 'avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'age_ever_married', 'age_hypertension', 'avg_glucose_level_hypertension', 'age_heart_disease', 'risk_count', 'urban']

In [48]:
# retained_columns = ['age_avg_glucose_level', 'avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'age_hypertension', 'avg_glucose_level_hypertension']

In [49]:
# retained_columns = ['age_avg_glucose_level', 'avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'age_hypertension', 'avg_glucose_level_hypertension', 'age_heart_disease']

In [50]:
# retained_columns = ['log_avg_glucose_level', 'age', 'log_bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban']

In [51]:
# retained_columns = ['log_avg_glucose_level', 'age', 'log_bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban', 'male']

In [52]:
# retained_columns = ['log_avg_glucose_level', 'age', 'log_bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban', 'male', 'work_type_private', 'smoking_status_never_smoked', 'work_type_self_employed']

In [53]:
# retained_columns = ['log_avg_glucose_level', 'age', 'log_bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban', 'male', 'work_type_private']

In [54]:
retained_columns = ['avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban', 'male', 'work_type_private']

In [55]:
X_train_retained = X_train[retained_columns]

In [56]:
X_val_retained = X_val[retained_columns]

### Create grid for hyperparameter tuning with GridSearchCV

In [57]:
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

In [58]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [59]:
grid_rf = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_rf,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

### Create optimizing function for hyperparameter tuning with Optuna

In [60]:
MIN_PRECISION = 0.15

In [61]:
# def objective_rf(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 50, 500),
#         'max_depth': trial.suggest_int('max_depth', 3, 50),
#         'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 50, 300),
#         'min_samples_split': trial.suggest_int('min_samples_split', 2, 40),
#         'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 15),
#         'min_weight_fraction_leaf': trial.suggest_float('min_weight_fraction_leaf', 0.0, 0.05),
#         'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7]),
#         'max_samples': trial.suggest_float('max_samples', 0.6, 1.0),
#         'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']), 
#         'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.005),
#         'ccp_alpha': trial.suggest_float('ccp_alpha', 0.0, 0.005),
#         'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample']),
#         'random_state': 42,
#         'n_jobs': -1
#     }
    
#     model = RandomForestClassifier(**params)
    
#     model.fit(X_train_retained, y_train)
    
#     y_pred_val = model.predict(X_val_retained)
    
#     recall = recall_score(y_val, y_pred_val)
#     precision = precision_score(y_val, y_pred_val)
#     f2_score = fbeta_score(y_val, y_pred_val, beta=2)
    
#     if precision < MIN_PRECISION:
#         return 0
#     # return recall + (precision * 0.01)
#     return f2_score + recall * 0.001 + precision * 0.00001

### Hyperparameter tuning  

In [62]:
# grid_rf.fit(X_train_retained, y_train)

In [63]:
# study_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
# study_rf.optimize(objective_rf, n_trials=500, show_progress_bar=True)

In [64]:
# best_params_rf = grid_rf.best_params_

In [65]:
# best_params_rf = study_rf.best_params

In [ ]:
best_params_rf = {'n_estimators': 221, 'max_depth': 24, 'max_leaf_nodes': 117, 'min_samples_split': 24, 'min_samples_leaf': 2, 'min_weight_fraction_leaf': 0.014351282275609447, 'max_features': 0.7, 'max_samples': 0.7350710798701602, 'criterion': 'entropy', 'min_impurity_decrease': 0.0028242087483375524, 'ccp_alpha': 0.002044925561392647, 'class_weight': 'balanced'}

In [67]:
print(best_params_rf)

{'n_estimators': 221, 'max_depth': 24, 'max_leaf_nodes': 117, 'min_samples_split': 24, 'min_samples_leaf': 2, 'min_weight_fraction_leaf': 0.014351282275609447, 'max_features': 0.7, 'max_samples': 0.7350710798701602, 'criterion': 'entropy', 'min_impurity_decrease': 0.0028242087483375524, 'ccp_alpha': 0.002044925561392647, 'class_weight': 'balanced'}


In [68]:
final_classifier = RandomForestClassifier(**best_params_rf, random_state=42)

In [69]:
final_classifier.fit(X_train_retained, y_train)

RandomForestClassifier(ccp_alpha=0.002044925561392647, class_weight='balanced',
                       criterion='entropy', max_depth=24, max_features=0.7,
                       max_leaf_nodes=117, max_samples=0.7350710798701602,
                       min_impurity_decrease=0.0028242087483375524,
                       min_samples_leaf=2, min_samples_split=24,
                       min_weight_fraction_leaf=0.014351282275609447,
                       n_estimators=221, random_state=42)

In [70]:
y_pred_val = final_classifier.predict(X_val_retained)

In [71]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [72]:
print(confusion_matrix(y_val, y_pred_val))

[[603 126]
 [ 10  27]]


In [73]:
print(accuracy_score(y_val, y_pred_val))

0.8224543080939948


In [74]:
print(precision_score(y_val, y_pred_val))

0.17647058823529413


In [75]:
print(recall_score(y_val, y_pred_val))

0.7297297297297297


In [76]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.4485049833887043


### Threshold tuning

In [77]:
best_threshold = 0.5

In [78]:
best_precision_score = 0

In [79]:
best_recall_score = 0

In [80]:
best_f2_score = 0

In [81]:
for threshold in np.arange(0.1, 0.9, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.3244274809160305
Threshold 0.15000000000000002 has F2 score 0.3395061728395062
Threshold 0.20000000000000004 has F2 score 0.3485838779956427
Threshold 0.25000000000000006 has F2 score 0.3621495327102804
Threshold 0.30000000000000004 has F2 score 0.3778337531486146
Threshold 0.3500000000000001 has F2 score 0.3978779840848806
Threshold 0.40000000000000013 has F2 score 0.39886039886039887
Threshold 0.45000000000000007 has F2 score 0.41033434650455924
Threshold 0.5000000000000001 has F2 score 0.4485049833887043
Threshold 0.5500000000000002 has F2 score 0.4411764705882353
Threshold 0.6000000000000002 has F2 score 0.44534412955465585
Threshold 0.6500000000000001 has F2 score 0.4329004329004329
Threshold 0.7000000000000002 has F2 score 0.35545023696682465
Threshold 0.7500000000000002 has F2 score 0.2925531914893617
Threshold 0.8000000000000002 has F2 score 0.18181818181818182
Threshold 0.8500000000000002 has F2 score 0.03333333333333333


In [82]:
print(best_threshold)

0.5000000000000001


In [83]:
print(best_recall_score)

0.7297297297297297


### Retrain on training + validation set

In [84]:
X_train_full = pd.concat([X_train_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

In [85]:
final_classifier.fit(X_train_full, y_train_full)

RandomForestClassifier(ccp_alpha=0.002044925561392647, class_weight='balanced',
                       criterion='entropy', max_depth=24, max_features=0.7,
                       max_leaf_nodes=117, max_samples=0.7350710798701602,
                       min_impurity_decrease=0.0028242087483375524,
                       min_samples_leaf=2, min_samples_split=24,
                       min_weight_fraction_leaf=0.014351282275609447,
                       n_estimators=221, random_state=42)

### Evaluate on test set

In [86]:
X_test_retained = X_test[retained_columns]

In [87]:
y_pred_test = final_classifier.predict(X_test_retained)

In [88]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [89]:
print(confusion_matrix(y_test, y_pred_test))

[[593 136]
 [  7  31]]


In [90]:
print(accuracy_score(y_test, y_pred_test))

0.8135593220338984


In [91]:
print(precision_score(y_test, y_pred_test))

0.18562874251497005


In [92]:
print(recall_score(y_test, y_pred_test))

0.8157894736842105


In [93]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.48589341692789967


### Test with best threshold 0.4

In [94]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [95]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[593 136]
 [  7  31]]


In [96]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.8135593220338984


In [97]:
print(precision_score(y_test, y_pred_test_threshold))

0.18562874251497005


In [98]:
print(recall_score(y_test, y_pred_test_threshold))

0.8157894736842105


In [ ]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.48589341692789967
